# 实践 1：一个小型客服聊天机器人
**学生版 notebook | 通过本地 Ollama 运行 Llama 3.2 1B**

用户在文本框中输入请求，然后按下**发送**。这个聊天机器人可以：

| 请求 | 预期回复来源 |
|---|---|
| `Track ORD-1001` | SQLite 返回包裹状态为 **shipped** |
| `Track ORD-1002` | SQLite 返回包裹状态为 **processing** |
| `How long does delivery take?` | 已存储的政策：**3-5 个工作日** |
| `How can I contact support?` | 已存储的联系方式：**support@example.com** |
| `What is a Python list?` | 模型生成的简短回答 |

你需要完成**三项任务**：**A** 编写提示词，**B** 解析模型输出，
以及 **C** 在需要时调用服务。数据加载器、两个辅助函数和用户界面已经提供。
所有订单记录均为虚构数据。这个聊天机器人无法修改订单或办理退款。

### 对话规则：每次提交一个完整请求
每次提交都会重新开始，系统不会记住之前的消息或订单 ID。
如果回复要求提供 ID，请重新发送完整请求 **`Track ORD-1001`**，而不是只发送 ID。

模型做出一次决策，Python 对其进行验证、选择回复来源，然后流程结束。
这是一个固定工作流。后续版本可以加入对话状态保存功能。

### 开始之前
解压 ZIP 文件，并从该文件夹启动 Jupyter。请使用现有的课程环境：
Python、Jupyter、`ipywidgets`，以及通过 Ollama 运行的指定模型 **`llama3.2:1b`**。
之前的准备环节同样会配置这些先决条件。如有需要，请在终端中运行：
`python -m pip install jupyterlab ipywidgets`，然后运行 `ollama pull llama3.2:1b`。
尝试实时请求前，请先打开 Ollama。不需要付费 API 或 GPU。

请按顺序运行。学生版会在每个尚未完成的任务检查处有意停止。
完整版可以运行所有单元格，并且在你启用可选的提示词测试或按下**发送**之前，
不会连接 Ollama。请将本项目保存在独立文件夹中：第 1 部分会根据 JSON 文件
重建本地 **demo** 数据库中的两个表。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/7608LLM/COMP7608B_Handson_1/

Mounted at /content/drive
/content/drive/MyDrive/7608LLM/COMP7608B_Handson_1


In [ ]:
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!nohup ollama serve > /tmp/ollama.log 2>&1 &

In [ ]:
!python -m pip install jupyterlab ipywidgets

In [ ]:
!curl http://127.0.0.1:11434/api/version
!ollama pull llama3.2:1b
!ollama list

{"version":"0.34.4"}
NAME           ID              SIZE      MODIFIED               
llama3.2:1b    baf6a787fdff    1.3 GB    Less than a second ago    


## 第 1 部分：加载数据
`data.json` 是可编辑的数据源。此代码块会将所有 `data*.json` 文件加载到
SQLite：`orders(order_id, status)` 和 `info(name, text)`。

若要添加包裹，请编辑 `data.json`，或在同一目录中新建 `data_extra.json`，例如：
```json
{"orders": [{"order_id": "ORD-1004", "status": "processing"}]}
```
编辑文件后，请**重新运行此代码块**。它会重建两个演示数据表，因此被删除的
记录也会消失。文件按文件名顺序加载；后加载的文件会替换具有相同 ID 或信息名称的记录。
打印出的数据行展示了聊天机器人能够读取的内容。


In [ ]:
import json, re, sqlite3
from pathlib import Path
from urllib import request

ROOT = Path.cwd()
data_files = sorted(ROOT.glob("data*.json"))
assert ROOT / "data.json" in data_files, "请解压资料包，并在解压后的文件夹中打开 Jupyter。"
if "db" in globals():
    db.close()
db = sqlite3.connect(ROOT / "support.sqlite3")
with db:
    db.execute("CREATE TABLE IF NOT EXISTS orders (order_id TEXT PRIMARY KEY, status TEXT)")
    db.execute("CREATE TABLE IF NOT EXISTS info (name TEXT PRIMARY KEY, text TEXT)")
    # 每次运行时，都根据文件重建这些演示数据表。
    db.execute("DELETE FROM orders")
    db.execute("DELETE FROM info")
    for path in data_files:
        data = json.loads(path.read_text(encoding="utf-8"))
        db.executemany("INSERT OR REPLACE INTO orders VALUES (?, ?)",
                       [(row["order_id"], row["status"]) for row in data.get("orders", [])])
        db.executemany("INSERT OR REPLACE INTO info VALUES (?, ?)", data.get("info", {}).items())
print("已加载的文件：", [p.name for p in data_files])
print("订单：", db.execute("SELECT * FROM orders ORDER BY order_id").fetchall())
print("信息：", db.execute("SELECT * FROM info ORDER BY name").fetchall())


已加载的文件： ['data.json']
订单： [('ORD-1001', 'shipped'), ('ORD-1002', 'processing'), ('ORD-1003', 'delivered')]
信息： [('contact_support', 'For further support, email support@example.com.'), ('delivery_info', 'Delivery normally takes 3-5 working days.')]


## 第 2 部分：辅助函数
请直接运行此代码块，不要修改。你可以在任务中调用这些函数：

| 函数 | 输入 | 返回值 |
|---|---|---|
| `ask_model(user_text)` | 当前请求；使用任务 A 中的 `SYSTEM_PROMPT` | Ollama 返回的原始 JSON 文本 |
| `call_service(intent, user_text)` | 服务标签和原始请求 | 从 SQLite 读取的回复文本，或要求用户澄清的信息 |
| `json.loads(raw)`（Python 标准库） | JSON 文本 | Python 值；无效 JSON 会引发 `ValueError` |

`call_service` 支持 `order_status`、`delivery_info` 和 `contact_support`。
它已经能够处理缺少 ID、包含多个 ID 和未知订单的情况。只有当用户提供 `ORD-`
后接四位数字时，它才会识别出 ID。你不需要编写 SQL。

`ask_model` 会发送两条消息：提示词和当前请求。JSON 模式会要求模型返回 JSON，
但你的解析器仍然必须检查其中的字段。CPU 设置为 `num_gpu=0`。
120 秒的套接字超时时间并不代表预期的响应延迟。


In [ ]:
OUTPUT_SCHEMA = {
    "oneOf": [
        {
            # 服务请求：由应用程序提供回复。
            "type": "object",
            "properties": {
                "intent": {
                    "type": "string",
                    "enum": [
                        "order_status",
                        "delivery_info",
                        "contact_support"
                    ]
                }
            },
            "required": ["intent"],
            "additionalProperties": False
        },
        {
            # 一般聊天：由模型提供回复。
            "type": "object",
            "properties": {
                "intent": {
                    "type": "string",
                    "enum": ["chat"]
                },
                "reply": {
                    "type": "string",
                    "minLength": 1
                }
            },
            "required": ["intent", "reply"],
            "additionalProperties": False
        }
    ]
}

In [ ]:
MODEL = "llama3.2:1b"
BASE_URL = "http://127.0.0.1:11434"
INTENTS = ["order_status", "delivery_info", "contact_support", "chat"]

def ask_model(user_text):
    """将此请求和任务 A 的提示词发送给 Ollama，并返回原始 JSON 文本。"""
    payload = {"model": MODEL, "stream": False, "format": OUTPUT_SCHEMA,
               "messages": [{"role": "system", "content": SYSTEM_PROMPT},
                            {"role": "user", "content": user_text}],
               "options": {"temperature": 0, "num_ctx": 2048, "num_predict": 192, "num_gpu": 0}}
    req = request.Request(BASE_URL + "/api/chat", data=json.dumps(payload).encode("utf-8"),
                          headers={"Content-Type": "application/json"})
    with request.urlopen(req, timeout=120) as response:
        result = json.load(response)
    if result.get("done") is not True or result.get("done_reason") == "length":
        raise ValueError("模型响应不完整。请检查 Ollama 和输出长度限制。")
    return result["message"]["content"]

def call_service(intent, user_text=""):
    """读取 SQLite 并返回回复文本。模型无法修改这些记录。"""
    if intent == "order_status":
        ids = list(dict.fromkeys(re.findall(r"(?<![A-Za-z0-9_])ORD-[0-9]{4}(?![A-Za-z0-9_])", user_text)))
        if not ids:
            return "请重新发送包含 ID 的完整请求，例如：Track ORD-1001。"
        if len(ids) > 1:
            return "请每次只查询一个订单。"
        row = db.execute("SELECT status FROM orders WHERE order_id = ?", (ids[0],)).fetchone()
        return f"订单 {ids[0]} 的状态是 {row[0]}。" if row else "找不到该订单。请查询客服联系方式。"
    if intent not in {"delivery_info", "contact_support"}:
        raise ValueError("未知服务。")
    row = db.execute("SELECT text FROM info WHERE name = ?", (intent,)).fetchone()
    return row[0] if row else "演示数据中没有此信息。"

print("两个辅助函数已就绪：ask_model 和 call_service。")


两个辅助函数已就绪：ask_model 和 call_service。


## 任务 A：编写提示词
编辑 **`prompt_student.txt`**，然后重新运行下面的提示词加载代码块。

| 意图 | 何时选择 | 模型输出 |
|---|---|---|
| `order_status` | 查询特定包裹，即使没有提供 ID | `{"intent":"order_status"}` |
| `delivery_info` | 查询一般配送政策或配送时间 | `{"intent":"delivery_info"}` |
| `contact_support` | 联系人工客服，或请求本演示无法执行的修改或退款 | `{"intent":"contact_support"}` |
| `chat` | 问候或与客服无关的问题 | `{"intent":"chat","reply":"简短回答"}` |

请定义这些标签并添加示例。要求模型只返回 JSON。服务请求的响应只能包含
`intent`；具体事实由服务提供。聊天响应还必须包含非空的 `reply`。要求模型
对一般问题给出简短回答，并承认自己不知道的内容。如果一条消息包含多个服务请求，
请让用户每次只提出一个请求。


In [ ]:
SYSTEM_PROMPT = (ROOT / "prompt_student.txt").read_text(encoding="utf-8")


In [ ]:
# 检查你是否已经替换初始说明。
assert SYSTEM_PROMPT.strip() and "TODO" not in SYSTEM_PROMPT
print("提示词已加载。请检查提示词文件中的文字和示例。")

RUN_PROMPT_TEST = True  # Ollama 就绪后可设为 True；这会真实调用一次模型。
if RUN_PROMPT_TEST:
    raw = ask_model("How many days does delivery normally take?")
    print("模型输出：", raw)
    assert json.loads(raw).get("intent") == "delivery_info"
    print("此请求选择了正确的意图。也请尝试其他表达方式。")
else:
    print("已跳过实时提示词测试。成功加载提示词并不能证明其质量。")


提示词已加载。请检查提示词文件中的文字和示例。
模型输出： {"intent":"delivery_info"}
此请求选择了正确的意图。也请尝试其他表达方式。


## 任务 B：解析并检查输出
实现 **`parse_output(raw)`**。该函数接收文本并返回一个 Python 字典。

1. 使用 `json.loads` 解析文本。
2. 检查结果是否为字典，并且其中的 `intent` 是否属于 `INTENTS`。
3. 服务决策必须恰好只有 `intent` 键。聊天决策必须恰好包含 `intent` 和 `reply`，
   并且 `reply` 必须是非空文本。
4. 对无效输出引发 `ValueError`；否则返回该字典。

下面的八项检查使用预先写好的字符串，因此不需要运行模型。
即使标签本身有效，也仍有可能错误理解用户请求。


In [25]:
def parse_output(raw):
    decision = json.loads(raw)
    if not isinstance(decision, dict):
        raise ValueError("模型输出必须是一个 JSON 对象。")

    intent = decision.get("intent")
    if intent not in INTENTS:
        raise ValueError("模型输出包含无效的 intent。")

    if intent == "chat":
        if set(decision) != {"intent", "reply"}:
            raise ValueError("chat 输出必须只包含 intent 和 reply。")
        reply = decision["reply"]
        if not isinstance(reply, str) or not reply.strip():
            raise ValueError("chat 的 reply 必须是非空字符串。")
    elif set(decision) != {"intent"}:
        raise ValueError("服务输出必须只包含 intent。")

    return decision


In [26]:
# 两个有效响应。
assert parse_output('{"intent":"order_status"}') == {"intent": "order_status"}
assert parse_output('{"intent":"chat","reply":"Hello!"}')["reply"] == "Hello!"

# 六个无效响应：每一个都必须引发 ValueError。
bad_outputs = [
    'not JSON', '{"intent":"refund"}', '{"intent":"chat"}',
    '{"intent":"chat","reply":7}', '{"intent":"chat","reply":""}',
    '{"intent":"order_status","reply":"Your order is delivered."}',
]
for raw in bad_outputs:
    try:
        parse_output(raw)
    except ValueError:
        continue
    raise AssertionError(f"应当拒绝：{raw}")
print("任务 B：已通过 8 项解析器检查，未运行模型。")


任务 B：已通过 8 项解析器检查，未运行模型。


## 任务 C：生成回复并连接工作流

最后一项任务是将模型、你的解析器和提供的服务
连接到聊天机器人界面。

对于每个提交的请求，应用程序会执行以下流程：

**用户请求 → 模型输出 → 验证后的决策 → 回复或服务调用 → 显示结果**

### C1. 实现 `make_reply(decision, user_text)`

`decision` 字典已经通过任务 B 中的验证。

- 如果意图是 `chat`，返回字典中包含的回复。
- 对于 `order_status`、`delivery_info` 或 `contact_support`，调用已提供的
  `call_service(intent, user_text)` 并返回其结果。

将原始 `user_text` 传给服务，以便服务查找订单 ID。
该服务已经能够处理缺失或未知的 ID，并从 SQLite 读取信息。
你不需要编写 SQL，也不需要自己提取 ID。

你的函数必须返回一个字符串，不应打印内容或再次调用模型。



In [ ]:
def make_reply(decision, user_text):
    """根据验证后的决策返回一条回复字符串。"""
    intent = decision["intent"]
    if intent == "chat":
        return decision["reply"]
    return call_service(intent, user_text)

### C2. 完成请求处理流程

在已提供的 UI 代码中，完成 `send_request` 内的 TODO。

1. 使用用户请求调用模型，并保存其原始输出。
2. 将该输出传给你在任务 B 中编写的解析器。
3. 将验证后的决策和原始请求传给 `make_reply`。

已提供的代码会显示回复并捕获错误。如果前面的某个步骤失败，
后续步骤就不应继续运行。

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# 已提供：创建界面。
text_box = widgets.Textarea(
    value="Track ORD-1001",
    description="请求：",
    layout=widgets.Layout(width="95%")
)
send_button = widgets.Button(
    description="发送",
    button_style="primary"
)
output = widgets.Output()


def send_request(button):
    user_text = text_box.value.strip()

    with output:
        output.clear_output(wait=True)

        if not user_text:
            print("请输入请求。")
            return

        send_button.disabled = True

        try:
            raw = ask_model(user_text)
            decision = parse_output(raw)
            reply = make_reply(decision, user_text)

            # 已提供：显示最终回复。
            print("助手：", reply)

        except (ValueError, OSError, KeyError, TypeError, RuntimeError) as exc:
            print("无法处理该请求，请重试。")
            print("调试信息：", exc)

        finally:
            send_button.disabled = False


# 已提供：连接按钮并显示界面。
send_button.on_click(send_request)
display(text_box, send_button, output)

## 完成
**完成标准：** 任务 A 的提示词已经完成，任务 B 和 C 中记录的 16 项检查全部通过，
并且你已经尝试过实时 UI。如果模型不可用，应将其记录为“未测试”。

在 `data_extra.json` 中添加 `ORD-1004`，重新运行第 1 部分，然后查询其状态。你不需要
修改辅助函数或任务 C。请保留已提供的记录，确保基础测试仍然有效。

讨论一个模型选择了错误意图的例子。解析器是否发现了这个错误？
如果要让后续消息只提供 ID，必须保留哪些状态？
哪些回复来自 SQLite，哪些依赖模型自身的知识？

这是一个教学原型。正确的 JSON 结构并不能保证路由正确，也不能保证一般问题的回答正确。
下一讲可以加入对话状态和更丰富的依据。

已提供调用器的参考资料：[Ollama 聊天 API](https://docs.ollama.com/api/chat)、
[JSON 输出](https://docs.ollama.com/capabilities/structured-outputs)、
[Llama 3.2 1B](https://ollama.com/library/llama3.2:1b)。
